# Module 9 — Embeddings from First Principles

Companion notebook to [`docs/modules/09_embeddings_from_first_principles.md`](../docs/modules/09_embeddings_from_first_principles.md).

This module runs for real, unlike most notebooks since Module 6 — `FakeEmbedder` is a genuine
bag-of-words hashing embedder (SHA-256 feature hashing, "the hashing trick"), not a mock. Every
number in this notebook is computed by real code, not a live neural embedding model (this machine
has no LLM/embedding runtime installed — see the repo's machine constraint). `OllamaEmbedder` and
`SentenceTransformersEmbedder` are built and unit-tested (with dependency injection / mock
transports) but honest-skipped here since they need a runtime or downloaded weights this machine
doesn't have.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_09"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Normalization, cosine similarity, and Matryoshka truncation

In [2]:
import numpy as np

from local_ai_rag.embeddings.embedder import normalize, cosine_similarity, truncate_embedding

a = np.array([3.0, 4.0])
print("normalize([3, 4]) ->", normalize(a), "(unit length:", np.linalg.norm(normalize(a)), ")")

zero = np.array([0.0, 0.0])
print("normalize(zero) ->", normalize(zero), "(zero-vector guard, no NaN)")

v1 = np.array([1.0, 0.0, 0.0])
v2 = np.array([1.0, 0.0, 0.0])
v3 = np.array([0.0, 1.0, 0.0])
print("cosine_similarity(identical) ->", cosine_similarity(v1, v2))
print("cosine_similarity(orthogonal) ->", cosine_similarity(v1, v3))

full = np.array([1.0, 2.0, 3.0, 4.0])
truncated = truncate_embedding(full, 2)
print("truncate_embedding(4d -> 2d) ->", truncated, "(unit length:", np.linalg.norm(truncated), ")")


normalize([3, 4]) -> [0.6 0.8] (unit length: 1.0 )
normalize(zero) -> [0. 0.] (zero-vector guard, no NaN)
cosine_similarity(identical) -> 1.0
cosine_similarity(orthogonal) -> 0.0
truncate_embedding(4d -> 2d) -> [0.4472136  0.89442719] (unit length: 0.9999999999999999 )


## 2. FakeEmbedder: a real (if crude) embedding technique — Lab 1

In [3]:
from local_ai_rag.embeddings.fake import FakeEmbedder

embedder = FakeEmbedder(dimensions=64)

cat_vec = await embedder.embed_query("the cat sat on the mat")
dog_vec = await embedder.embed_query("a dog sat on a rug")
space_vec = await embedder.embed_query("distant galaxies and stars")

print("cat vs dog (share 'sat', 'on', 'a'):", cosine_similarity(cat_vec, dog_vec))
print("cat vs space (no shared words):     ", cosine_similarity(cat_vec, space_vec))
print()
print("This is the real, measurable property that makes FakeEmbedder useful for testing:")
print("texts that share words score higher than texts that don't -- not a fabricated number.")


cat vs dog (share 'sat', 'on', 'a'): 0.25000000000000006
cat vs space (no shared words):      -0.1767766952966369

This is the real, measurable property that makes FakeEmbedder useful for testing:
texts that share words score higher than texts that don't -- not a fabricated number.


## 3. Building a corpus, storing it, and brute-force search — Labs 1-3

In [4]:
from generate_and_search import CORPUS, build_index

index = await build_index(embedder)
print(f"Indexed {len(index)} documents")

query_vec = await embedder.embed_query("I forgot my password")
results = index.search(query_vec, k=3)
for r in results:
    print(f"{r.score:.3f}  {r.doc_id:15s}  {r.text}")


Indexed 5 documents
0.316  doc_password     How to reset your password
0.267  doc_password2    Forgot password recovery steps for your account
0.000  doc_billing      Update your billing information and payment method


## 4. Metadata filtering — Lab 6

In [5]:
filtered = index.search(query_vec, k=5, metadata_filter={"category": "account"})
for r in filtered:
    print(f"{r.score:.3f}  {r.doc_id:15s}  category={r.metadata['category']}")


0.316  doc_password     category=account
0.267  doc_password2    category=account


## 5. Evaluating recall@k, precision@k, MRR, nDCG@k — Lab 4

In [6]:
from generate_and_search import EVAL_CASES
from local_ai_rag.embeddings.eval import evaluate_embedder

summary = await evaluate_embedder(embedder, index, EVAL_CASES, k=3)
print(f"mean recall@3:    {summary.mean_recall_at_k:.2f}")
print(f"mean precision@3: {summary.mean_precision_at_k:.2f}")
print(f"MRR:              {summary.mrr:.2f}")
print(f"mean nDCG@3:      {summary.mean_ndcg_at_k:.2f}")
print(f"mean latency (s): {summary.mean_query_latency_seconds:.6f}")


mean recall@3:    1.00
mean precision@3: 0.50
MRR:              1.00
mean nDCG@3:      1.00
mean latency (s): 0.000084


## 6. Comparing two embedding configurations — Lab 5

In [7]:
from compare_embedding_models import compare_embedders, results_to_markdown_table

comparison = await compare_embedders(
    {"fake-64d": FakeEmbedder(dimensions=64), "fake-4d (severe hash collisions)": FakeEmbedder(dimensions=4)},
    k=3,
)
print(results_to_markdown_table(comparison))
print("Lower dimensionality -> more hash collisions -> worse ranking quality (MRR, nDCG drop),")
print("even though recall@k can still look fine. This is a real, measured dimensionality effect.")


# Lab 5 — compare two embedding models

(corpus size: 5, eval cases: 2)

| Model | Recall@k | Precision@k | MRR | nDCG@k |
|---|---:|---:|---:|---:|
| fake-64d | 1.00 | 0.50 | 1.00 | 1.00 |
| fake-4d (severe hash collisions) | 1.00 | 0.50 | 0.42 | 0.60 |

Lower dimensionality -> more hash collisions -> worse ranking quality (MRR, nDCG drop),
even though recall@k can still look fine. This is a real, measured dimensionality effect.


## 7. Embedding throughput — docs/sec

In [8]:
from local_ai_rag.embeddings.eval import measure_embedding_throughput

texts = [text for text, _ in CORPUS.values()] * 20  # 100 short docs
throughput = await measure_embedding_throughput(embedder, texts)
print(f"FakeEmbedder throughput: {throughput:.1f} docs/sec (over {len(texts)} docs)")


FakeEmbedder throughput: 98627.5 docs/sec (over 100 docs)


## 8. Real embedder adapters — honest skip

`OllamaEmbedder` (calls a running Ollama server's `/api/embeddings`) and
`SentenceTransformersEmbedder` (wraps `sentence-transformers` models like BGE/GTE) are both built
and fully unit-tested — `OllamaEmbedder` against `httpx.MockTransport`, `SentenceTransformersEmbedder`
against injected fake `load_fn`/`encode_fn`. Neither is exercised here because this machine has no
Ollama server running and no `sentence-transformers` model downloaded (see the repo's machine
constraint: no LLM runtime or model weights on this Mac). On the target 32GB Mac, both classes can
be used as drop-in `Embedder` implementations with the exact same `NumpyEmbeddingIndex` and
`eval.py` code shown above — no code in this notebook would need to change.

In [9]:
import shutil

ollama_available = shutil.which("ollama") is not None
print(f"ollama CLI on PATH: {ollama_available}")
print("Skipping real-model cells: this development machine intentionally has no LLM/embedding runtime installed.")


ollama CLI on PATH: False
Skipping real-model cells: this development machine intentionally has no LLM/embedding runtime installed.
